In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    CharacterTextSplitter ,
    RecursiveCharacterTextSplitter ,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
)
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from huggingface_hub import login
from langchain_huggingface import HuggingFaceEmbeddings


c:\Users\ASUS\Desktop\scientific-paper-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ASUS\AppData\Local\Temp\ipykernel_22704\572718368.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [2]:
load_dotenv()

_token = os.getenv("HF_TOKEN")
if _token:
    login(token=_token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# How many characters in all the documents?

knowledge_base_path = "knowledge_base/**/*.md"

files = glob.glob(knowledge_base_path, recursive=True)
print(files)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")


['knowledge_base\\company\\about.md', 'knowledge_base\\company\\careers.md', 'knowledge_base\\company\\culture.md', 'knowledge_base\\company\\overview.md', 'knowledge_base\\contracts\\Contract with Advantage Medical Coverage for Healthllm.md', 'knowledge_base\\contracts\\Contract with Apex Reinsurance for Rellm - AI-Powered Enterprise Reinsurance Solution.md', 'knowledge_base\\contracts\\Contract with Atlantic Risk Solutions for Bizllm.md', 'knowledge_base\\contracts\\Contract with Belvedere Insurance for Markellm.md', 'knowledge_base\\contracts\\Contract with BrightWay Solutions for Markellm.md', 'knowledge_base\\contracts\\Contract with ConnectInsure Agency for Markellm.md', 'knowledge_base\\contracts\\Contract with Continental Commercial Group for Bizllm.md', 'knowledge_base\\contracts\\Contract with DriveSmart Insurance for Carllm.md', 'knowledge_base\\contracts\\Contract with Evergreen Life Insurance for Lifellm.md', 'knowledge_base\\contracts\\Contract with EverGuard Insurance fo

In [4]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("knowledge_base/*")


documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")



Loaded 76 documents


In [5]:
documents[0]


Document(metadata={'source': 'knowledge_base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a

In [6]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")


print(f"First chunk:\n\n{chunks[0]}")


Divided into 413 chunks
First chunk:

page_content='# About Insurellm

Insurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.

The company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.' metadata={'source': 'knowledge_base\\company\\about.md', 'doc_type': 'company'}


In [7]:
chunks[0]


Document(metadata={'source': 'knowledge_base\\company\\about.md', 'doc_type': 'company'}, page_content='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

In [8]:

print(len(chunks))

413


In [9]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db_name = "vector_store"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12693.21it/s]


In [10]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_database = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=db_name
)
print(f"Vector Store created with {vector_database._collection.count()} documents")


Vector Store created with 413 documents


In [12]:
collection = vector_database._collection
count = collection.count()

# Fetch one item, including its embedding vector
sample = collection.get(limit=1, include=["embeddings"])

# The embedding vector for that one item
sample_embedding = sample["embeddings"][0]

dimensions = len(sample_embedding)

print(f"Number of documents: {count}")
print(f"Number of dimensions: {dimensions}")


Number of documents: 413
Number of dimensions: 384
